# Rappels de produits — analyse exploratoire

Sujet 5, RappelConso V2. Source : data.economie.gouv.fr, producteurs DGCCRF,
DGAL, DGEC et DGPR, Licence Ouverte Etalab 2.0.

Ce notebook s'exécute de bout en bout après redémarrage du noyau (QC-02).
Il suppose que `python src/download.py` a déjà été lancé.

**Synthèse statistique** — à rédiger ici en fin de projet, une à deux pages,
avant le détail technique : un lecteur non spécialiste doit pouvoir s'arrêter
après cette section.

## 1. Chargement

Les deux fichiers bruts sont chargés tels quels, sans transformation : le
profil doit décrire la source, pas une version déjà nettoyée.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

fiches = pd.read_parquet(RACINE / "data" / "raw" / "rappelconso_fiches.parquet")
produits = pd.read_parquet(RACINE / "data" / "raw" / "rappelconso_produits.parquet")

manifeste = json.loads(
    (RACINE / "data" / "raw" / "rappelconso_fiches.manifeste.json").read_text()
)
print(f"Téléchargé le {manifeste['date_telechargement']}")
print(f"fiches   : {fiches.shape[0]} lignes, {fiches.shape[1]} colonnes")
print(f"produits : {produits.shape[0]} lignes, {produits.shape[1]} colonnes")

## 2. Profil du jeu de données (EF-02)

Dimensions, types, taux de valeurs manquantes par colonne, cardinalité des
variables catégorielles, période couverte.

In [ ]:
def profil(df: pd.DataFrame) -> pd.DataFrame:
    """Une ligne par colonne : type, remplissage, cardinalité, exemple."""
    return pd.DataFrame(
        {
            "type": df.dtypes.astype(str),
            "taux_vide_%": (df.isna().mean() * 100).round(1),
            "valeurs_distinctes": df.nunique(),
            "exemple": [
                str(df[c].dropna().iloc[0])[:50] if df[c].notna().any() else ""
                for c in df.columns
            ],
        }
    )


profil_fiches = profil(fiches)
profil_fiches

### Colonnes inexploitables en l'état

Une colonne vide à plus de 50 % ne peut pas porter une conclusion. Le seuil
retenu et le sort de chaque colonne concernée sont tracés dans
`docs/decisions.md`.

In [ ]:
profil_fiches[profil_fiches["taux_vide_%"] > 50].sort_values(
    "taux_vide_%", ascending=False
)

### Période couverte

Le mois en cours est incomplet : toute courbe temporelle qui l'inclut montre
une chute qui n'existe pas.

In [ ]:
dates = pd.to_datetime(fiches["date_publication"], errors="coerce", utc=True)
print(f"de {dates.min():%Y-%m-%d} à {dates.max():%Y-%m-%d}")
print(f"dates non interprétables : {dates.isna().sum()}")
dates.dt.year.value_counts().sort_index()

## 3. Contrôles qualité minimaux (section 8 du cahier des charges)

Chaque constat qui suit ouvre une entrée dans `docs/anomalies.md`.

### Doublons : une fiche révisée apparaît-elle plusieurs fois ?

In [ ]:
print(f"lignes                        : {len(fiches)}")
print(f"numero_fiche distincts        : {fiches['numero_fiche'].nunique()}")
print(f"couples fiche+version         : {fiches[['numero_fiche', 'numero_version']].drop_duplicates().shape[0]}")
print(f"lignes strictement identiques : {fiches.duplicated().sum()}")

fiches["numero_fiche"].value_counts().head(10)

### Valeurs manquantes déguisées

Un vide ne se présente pas toujours comme un `NaN` : chaîne vide, « non
renseigné », « - », « n/a ».

In [ ]:
DEGUISEMENTS = {"", " ", "-", "n/a", "na", "non renseigné", "non renseigne", "inconnu"}

for colonne in fiches.select_dtypes(include="object").columns:
    serie = fiches[colonne].astype(str).str.strip().str.lower()
    nombre = serie.isin(DEGUISEMENTS).sum()
    if nombre:
        print(f"{colonne:<45} {nombre:>6} vides déguisés")

### Champs multivalués (ANO-03)

Tout comptage sur ces colonnes avant éclatement est faux.

In [ ]:
for colonne, separateur in [
    ("conduites_a_tenir_par_le_consommateur", "|"),
    ("modeles_ou_references", "¤"),
]:
    touchees = fiches[colonne].astype(str).str.contains(separateur, regex=False).sum()
    print(f"{colonne:<42} {touchees:>6} lignes contiennent « {separateur} »")

### Variantes d'écriture des marques (question Q4)

Sans normalisation, un classement des marques est faux.

In [ ]:
marques = fiches["marque_produit"].dropna().astype(str)
print(f"marques distinctes, telles quelles : {marques.nunique()}")
print(f"après minuscules et espaces rognés : {marques.str.strip().str.lower().nunique()}")
marques.str.strip().str.lower().value_counts().head(15)

### Périmètre des deux fichiers (ANO-01)

Contrôle de la jointure EF-06 avant de l'écrire.

In [ ]:
cle = ["numero_fiche", "numero_version"]
cles_fiches = set(map(tuple, fiches[cle].astype(str).values))
cles_produits = set(map(tuple, produits[cle].astype(str).values))

print(f"clés dans fiches seulement   : {len(cles_fiches - cles_produits)}")
print(f"clés dans produits seulement : {len(cles_produits - cles_fiches)}")
print(f"clés communes                : {len(cles_fiches & cles_produits)}")

## 4. Distributions (EF-03)

À écrire — Feriel. Pour chaque variable quantitative retenue : tendance
centrale, dispersion, quartiles, et la justification du choix moyenne ou
médiane.

## 5. Agrégations (EF-04)

À écrire — Feriel. Une agrégation temporelle, une agrégation par catégorie, et
le croisement des deux. Les fonctions vivent dans `src/indicators.py`, pas ici.

## 6. Hypothèses (EF-05)

À écrire — Feriel. Deux hypothèses formulées **avant** calcul dans
`docs/questions.md`, puis testées ici et tranchées par écrit.